In [0]:
# 1. Configuration
catalog = "capstone6_telecom"
schema = "bronze"

volume_path = "/Volumes/capstone6_telecom/bronze/ext_raw_landing"

telemetry_path = volume_path + "/telemetry_data"
event_path = volume_path + "/event_data"

# Auto Loader state lives in a separate volume, outside the landing volume, so the job's file arrival trigger
# (which watches the landing volume recursively) is not fired by checkpoint / schema writes.
# Copied unchanged from ext_raw_landing/_checkpoints and _schemas on 2026-11-04, so already-loaded files are not reprocessed.
metadata_path = "/Volumes/capstone6_telecom/bronze/autoloader_metadata"
schema_root = metadata_path + "/schemas/telecom_bronze_v2"
checkpoint_root = metadata_path + "/checkpoints/telecom_bronze_v2"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
# 2. Define each feed
feeds = [
    {
        "name": "network_kpi_measurements",
        "path": telemetry_path,
        "file_filter": "network_kpi_measurements*.csv",
        "target": "bronze_network_kpi_measurements"
    },
    {
        "name": "network_outage_events",
        "path": event_path,
        "file_filter": "network_outage_events*.csv",
        "target": "bronze_network_outage_events"
    },
    {
        "name": "customer_complaints",
        "path": event_path,
        "file_filter": "customer_complaints*.csv",
        "target": "bronze_customer_complaints"
    }
]

In [0]:
# 3. Load the feeds using Auto Loader
from pyspark.sql import functions as F

for feed in feeds:

    feed_name = feed["name"]

    source_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_root + "/" + feed_name)
        .option("cloudFiles.inferColumnTypes", "false")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("header", "true")
        .option("pathGlobFilter", feed["file_filter"])
        .load(feed["path"]))

    bronze_df = (source_df
        .withColumn("_source_system", F.lit("ADLS Gen2"))
        .withColumn("_source_feed", F.lit(feed_name))
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp()))

    query = (bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_root + "/" + feed_name)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{schema}.{feed['target']}"))

    query.awaitTermination()
    print(f"Loaded {feed_name} into {feed['target']}")

In [0]:
# 4. Verify the three ADLS Bronze tables
for feed in feeds:
    print(f"Table: {feed['target']}")
    display(spark.table(f"{catalog}.{schema}.{feed['target']}").limit(5))